In [2]:
import sys
import copy
import json
import math
import random
import time
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")

pd.set_option(
    "display.max_columns",
    100,
)

pd.set_option(
    "display.width",
    160,
)


# --------------------------------------------------
# Locate project root
# --------------------------------------------------

current = Path.cwd()

PROJECT_ROOT = None

for path in [current] + list(current.parents):

    if (path / "src").is_dir():

        PROJECT_ROOT = path

        break


if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "Could not find project root containing 'src'."
    )


if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


print(
    "Project root:",
    PROJECT_ROOT,
)


# --------------------------------------------------
# Project imports
# --------------------------------------------------

from src.datasets.ecg_dataset import (
    PTBXLDataset,
)

from src.models.ecg.cnn1d import (
    CNN1D,
)

from src.utils.device import (
    get_device,
)

from src.utils.seed import (
    set_seed,
)


DEVICE = get_device()

set_seed(42)

print(
    "Device:",
    DEVICE,
)

Project root: d:\College Material\Major Project\CardioFusion-XAI\ml-service
Device: cuda


In [3]:
DATA_DIR = (
    PROJECT_ROOT
    / "data"
)

PREPROCESSED_DIR = (
    DATA_DIR
    / "processed"
    / "ecg"
    / "ptbxl"
)

PREPROCESSING_VERSION = (
    "100hz_record_zscore"
)

PROCESSED_DIR = (
    PREPROCESSED_DIR
    / PREPROCESSING_VERSION
)

MANIFEST_PATH = (
    PROCESSED_DIR
    / "manifests"
    / "ptbxl_manifest.csv"
)

PREPROCESSING_CONFIG_PATH = (
    PROCESSED_DIR
    / "preprocessing_config.json"
)


CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "ecg"
    / "ptbxl"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "metrics"
    / "ecg"
    / "cnn1d"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Processed data:",
    PROCESSED_DIR,
)

print(
    "Manifest:",
    MANIFEST_PATH,
)

print(
    "Checkpoint directory:",
    CHECKPOINT_DIR,
)

print(
    "Results directory:",
    RESULTS_DIR,
)

Processed data: d:\College Material\Major Project\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore
Manifest: d:\College Material\Major Project\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\manifests\ptbxl_manifest.csv
Checkpoint directory: d:\College Material\Major Project\CardioFusion-XAI\ml-service\checkpoints\ecg\ptbxl
Results directory: d:\College Material\Major Project\CardioFusion-XAI\ml-service\artifacts\metrics\ecg\cnn1d


In [9]:
project_search_roots = [
    PROCESSED_DIR,
    PREPROCESSED_DIR,
    DATA_DIR,
]

project_search_roots = [
    root
    for root in project_search_roots
    if root.is_dir()
]


if not MANIFEST_PATH.is_file():

    manifest_candidates = sorted(
        {
            path
            for root in project_search_roots
            for path in root.rglob("*.csv")
            if path.is_file()
        },
        key=lambda path: (
            0 if "manifest" in path.name.lower() else 1,
            0 if "ptbxl" in path.name.lower() or "ptb-xl" in path.name.lower() else 1,
            0 if PROCESSED_DIR in path.parents else 1,
            len(path.parts),
            str(path).lower(),
        ),
    )

    if not manifest_candidates:

        raise FileNotFoundError(
            "No PTB-XL CSV manifest was found under:\n"
            f"{DATA_DIR}\n\n"
            "Run the PTB-XL preprocessing step first."
        )

    MANIFEST_PATH = manifest_candidates[0]

    print(
        "Using discovered manifest:",
        MANIFEST_PATH,
    )


if not PREPROCESSING_CONFIG_PATH.is_file():

    config_candidates = sorted(
        {
            path
            for root in project_search_roots
            for path in root.rglob("preprocessing_config.json")
            if path.is_file()
        },
        key=lambda path: (
            0 if path.parent.name == PREPROCESSING_VERSION else 1,
            0 if "ptbxl" in str(path).lower() else 1,
            len(path.parts),
            str(path).lower(),
        ),
    )

    if not config_candidates:

        raise FileNotFoundError(
            "No preprocessing_config.json was found under:\n"
            f"{DATA_DIR}\n\n"
            "Run the PTB-XL preprocessing pipeline first."
        )

    PREPROCESSING_CONFIG_PATH = config_candidates[0]

    print(
        "Using discovered preprocessing configuration:",
        PREPROCESSING_CONFIG_PATH,
    )


with open(
    PREPROCESSING_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:

    preprocessing_config = json.load(f)


print(
    json.dumps(
        preprocessing_config,
        indent=2,
    )
)

FileNotFoundError: No preprocessing_config.json was found under:
d:\College Material\Major Project\CardioFusion-XAI\ml-service\data

Run the PTB-XL preprocessing pipeline first.

In [ ]:
ptbxl = pd.read_csv(
    MANIFEST_PATH,
)

print(
    "Manifest shape:",
    ptbxl.shape,
)

display(
    ptbxl.head()
)


DIAGNOSTIC_LABELS = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP",
]


diagnostic_targets = [
    label
    for label in DIAGNOSTIC_LABELS
    if label in ptbxl.columns
]


if len(diagnostic_targets) != 5:

    raise ValueError(
        "Expected all five PTB-XL diagnostic labels."
    )


print(
    "Diagnostic targets:",
    diagnostic_targets,
)


train_df = (
    ptbxl[
        ptbxl["split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    ptbxl[
        ptbxl["split"] == "val"
    ]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    ptbxl[
        ptbxl["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    f"Train      : {len(train_df):,}"
)

print(
    f"Validation : {len(val_df):,}"
)

print(
    f"Test       : {len(test_df):,}"
)

In [ ]:
train_patients = set(
    train_df["patient_id"]
)

val_patients = set(
    val_df["patient_id"]
)

test_patients = set(
    test_df["patient_id"]
)


train_val_overlap = (
    train_patients
    & val_patients
)

train_test_overlap = (
    train_patients
    & test_patients
)

val_test_overlap = (
    val_patients
    & test_patients
)


print(
    "Train ∩ Val :",
    len(train_val_overlap),
)

print(
    "Train ∩ Test:",
    len(train_test_overlap),
)

print(
    "Val ∩ Test  :",
    len(val_test_overlap),
)


if any([
    train_val_overlap,
    train_test_overlap,
    val_test_overlap,
]):

    raise RuntimeError(
        "Patient-level leakage detected."
    )


print(
    "Patient-level split integrity passed."
)

In [ ]:
train_dataset = PTBXLDataset(
    train_df,
    PROJECT_ROOT,
    diagnostic_targets,
    [],
)

val_dataset = PTBXLDataset(
    val_df,
    PROJECT_ROOT,
    diagnostic_targets,
    [],
)

test_dataset = PTBXLDataset(
    test_df,
    PROJECT_ROOT,
    diagnostic_targets,
    [],
)


print(
    "Train dataset:",
    len(train_dataset),
)

print(
    "Validation dataset:",
    len(val_dataset),
)

print(
    "Test dataset:",
    len(test_dataset),
)

In [ ]:
MODEL_CONFIG = {
    "name": "cnn1d",

    "input_channels": NUM_LEADS,

    "num_diagnostic_classes":
        NUM_DIAGNOSTIC,

    "dropout": 0.50,
}


print(
    json.dumps(
        MODEL_CONFIG,
        indent=2,
    )
)

In [ ]:
model = CNN1D(
    input_channels=MODEL_CONFIG[
        "input_channels"
    ],

    num_diagnostic_classes=MODEL_CONFIG[
        "num_diagnostic_classes"
    ],

    dropout=MODEL_CONFIG[
        "dropout"
    ],
)


model = model.to(
    DEVICE
)


print(model)


total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


print(
    f"\nTotal parameters    : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_parameters:,}"
)